<a href="https://colab.research.google.com/github/GanghyunShin02/Fenicsx/blob/main/HeatExchanger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# FEniCSx on Google Colab (default: DOLFINx 0.11.x)
# SeoulTechPSE/fenicsx-colab + OpenMPI 5.x (PRRTE) slot patch

from google.colab import drive
import os, multiprocessing, subprocess
from pathlib import Path

if not os.path.ismount('/content/drive'):
    drive.mount('/content/drive')
else:
    print('Google Drive already mounted')

REPO_URL = 'https://github.com/seoultechpse/fenicsx-colab.git'
REPO_DIR = Path('/content/fenicsx-colab')
if not REPO_DIR.exists():
    print('Cloning fenicsx-colab...')
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    print('Repository already exists')

# OpenMPI 5.x (PRRTE) slot patch
# Colab VM has 2 CPUs; force slots=4 via hostfile (oversubscribe)
N_PROC = 4
with open('/root/hostfile', 'w') as f:
    f.write(f'localhost slots={N_PROC}\n')
os.environ['OMPI_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
os.environ['PRTE_MCA_rmaps_default_mapping_policy'] = 'slot:OVERSUBSCRIBE'
print(f'MPI: {N_PROC} slots on {multiprocessing.cpu_count()} CPUs (oversubscribe enabled)')

DOLFINX_VERSION = '0.11'   # pin to '0.10' for legacy notebooks not yet migrated
USE_COMPLEX = False
USE_CLEAN   = False
ENV_NAME    = None         # leave None for default 'fenicsx'; set e.g. 'fenicsx010'
                            # to keep a 0.10 environment alongside the 0.11 default

opts = [f'--version {DOLFINX_VERSION}']
if USE_COMPLEX: opts.append('--complex')
if USE_CLEAN:   opts.append('--clean')
if ENV_NAME:    opts.append(f'--env-name {ENV_NAME}')
get_ipython().run_line_magic('run', f"{REPO_DIR / 'setup_fenicsx.py'} {' '.join(opts)}")

Mounted at /content/drive
Cloning fenicsx-colab...
MPI: 4 slots on 2 CPUs (oversubscribe enabled)
🔧 FEniCSx Setup Configuration
DOLFINx version : 0.11
PETSc type      : real
Env name        : fenicsx
Clean install   : False

📦 Google Drive detected — using persistent cache

🔧 Installing FEniCSx environment...

🔍 Verifying installation...
✅ Installed: DOLFINx 0.11.0
✅ Installed: Real PETSc (float64)

✨ Loading FEniCSx Jupyter magic... %%fenicsx registered (env: fenicsx)

✅ FEniCSx setup complete!

Next steps:
  1. Run %%fenicsx --info to verify installation
  2. Use %%fenicsx in cells to run FEniCSx code
  3. Use -np N for parallel execution (e.g., %%fenicsx -np 4)

📌 Note: DOLFINx 0.11 is installed in env 'fenicsx'
   - Real PETSc: recommended for most FEM problems
   - For complex problems, reinstall with --complex


In [ ]:
%%fenicsx -np 4
import numpy as np
import matplotlib.pyplot as plt

import dolfinx
from mpi4py import MPI
import gmsh
from dolfinx import fem
from dolfinx import mesh,io
from dolfinx.io import VTXWriter,gmsh as gmshio
import ufl
from ufl import (grad,
                 dot,
                 inner,
                 TrialFunction,
                 TestFunction)
from dolfinx.fem import Function

gmsh.initialize()

if MPI.COMM_WORLD.rank==0:
  Cylinder1=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,1)
  Cylinder2=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,1.1)
  Cylinder22=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,1.1)
  Cylinder3=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,2.1)
  Cylinder33=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,2.1)
  Cylinder4=gmsh.model.occ.addCylinder(0,0,2.2,10,0,0,2.2)

  inpipe,inflow=gmsh.model.occ.fragment([(3,Cylinder2)],[(3,Cylinder1)])
  outflow,_=gmsh.model.occ.fragment([(3,Cylinder3)],[(3,Cylinder22)])
  outpipe,_=gmsh.model.occ.fragment([(3,Cylinder4)],[(3,Cylinder33)])

  gmsh.model.occ.synchronize()

  gmsh.model.addPhysicalGroup(3,[inflow],1)
  gmsh.model.addPhysicalGroup(3,[inpipe],2)
  gmsh.model.addPhysicalGroup(3,[outflow],3)
  gmsh.model.addPhysicalGroup(3,[outpipe],4)

  gmsh.option.setNumber('Mesh.MeshSizeMax',0.5)
  gmsh.option.setNumber('Mesh.MeshSizeMin',0.1)
  gmsh.model.generate(3)

mesh_data=gmshio.model_to_mesh(gmsh.model,MPI.COMM_WORLD,0,3)
domain=mesh_data.mesh

facet_marker=mesh_data.facet_tags()
cell_marker=mesh_data.cell_tags()

gmsh.finalize()

with io.XDMFFile(domain.comm, "heat_exchanger3D_geometry2.xdmf", "w") as xdmf:
    xdmf.write_mesh(domain)
    domain.topology.create_connectivity(domain.topology.dim, domain.topology.dim)
    xdmf.write_meshtags(cell_marker, domain.geometry)